In [1]:
# File listing 
import os, numpy as np, pandas as pd
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/patch overlay/IU_PDA_T11/IU_PDA_T11_overlay.tiff
/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/patch overlay/IU_PDA_HM11/IU_PDA_HM11_overlay.tiff
/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/patch overlay/IU_PDA_T1/IU_PDA_T1_overlay.tiff
/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/patch overlay/IU_PDA_HM13/IU_PDA_HM13_overlay.tiff
/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/patch overlay/IU_PDA_T4/IU_PDA_T4_overlay.tiff
/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/h5 patches/IU_PDA_T3/IU_PDA_T3_patches.h5
/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/h5 patches/IU_PDA_T11/IU_PDA_T11_patches.h5
/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/h5 patches/IU_PDA_HM11/IU_PDA_HM11_patches.h5
/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/h5 patches/IU_PDA_T1/IU_PDA_T1_patches.h5
/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm

In [2]:
# Install 
# CONCH v1.5 uses timm directly — same as UNI2-h, NO custom conch package needed
!pip install -q timm huggingface_hub pillow h5py tqdm

In [4]:
# Auth 
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in to HuggingFace")

Logged in to HuggingFace


In [5]:
# Imports & Config 
import os, glob, torch
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

CONFIG = {
    "input_png_root": "/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/.png patches/.png patches",
    "output_root":    "/kaggle/working/Features/CONCHv1_5",
    "samples": ["IU_PDA_HM11", "IU_PDA_HM13", "IU_PDA_T1",
                "IU_PDA_T11",  "IU_PDA_T3",   "IU_PDA_T4"],
    "batch_size": 16,   # ← 16 not 32: 448×448 inputs are 4× larger than 224×224
}

os.makedirs(CONFIG["output_root"], exist_ok=True)
print(f"Device : {'CUDA' if torch.cuda.is_available() else 'CPU — enable GPU T4!'}")
print(f"Input  : {CONFIG['input_png_root']}")
print(f"Output : {CONFIG['output_root']}")

Device : CUDA
Input  : /kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/.png patches/.png patches
Output : /kaggle/working/Features/CONCHv1_5


In [10]:
# Model 
# CONCH v1.5 key differences from CONCH v1 and UNI2-h:
# 1. Uses timm (like UNI2-h), NOT the custom conch package (unlike CONCH v1)
# 2. Input size is 448×448, NOT 224×224 — BICUBIC interpolation required
# 3. Weights file is named pytorch_model_vision.bin (vision only, no text encoder)
# 4. Output dim is 768 (ViT-L), not 512 (v1) or 1536 (UNI2-h)
# 5. Mean pool over all patch tokens — no register tokens to skip

import timm
from huggingface_hub import hf_hub_download

class CONCHv15Extractor:
    def __init__(self):
        assert torch.cuda.is_available(), "Enable GPU T4 in Kaggle Settings!"

        # Download vision weights from HuggingFace
        # Note: file is pytorch_model_vision.bin — vision encoder only
        ckpt_path = hf_hub_download(
            repo_id="MahmoodLab/conchv1_5",
            filename="pytorch_model_vision.bin"
        )
        print(f"Weights downloaded to: {ckpt_path}")

        # Build ViT-L with 448×448 input
        self.model = timm.create_model(
            "vit_large_patch16_224",
            img_size=448,          # critical — must be 448, not 224
            patch_size=16,
            init_values=1.0,
            num_classes=0,         # remove classifier head
            dynamic_img_size=True
        )

        # Load weights manually (not pretrained=True, file has custom name)
        # NEW — strips trunk. prefix, ignores contrast head
        raw_state_dict = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        
        # Print keys to confirm structure (run once, then can remove)
        all_keys = list(raw_state_dict.keys())
        print(f"Total keys in file : {len(all_keys)}")
        print(f"Sample keys        : {all_keys[:5]}")
        
        # Strip "trunk." prefix — keep only vision backbone keys
        trunk_state_dict = {
            k.replace("trunk.", ""): v
            for k, v in raw_state_dict.items()
            if k.startswith("trunk.")          # only backbone keys
            and not k.startswith("trunk.fc")   # skip classifier if present
        }
        
        print(f"Trunk keys extracted: {len(trunk_state_dict)}")
        
        # strict=False to silently ignore any remaining mismatches
        missing, unexpected = self.model.load_state_dict(trunk_state_dict, strict=False)
        print(f"Missing keys   : {len(missing)}    (should be 0)")
        print(f"Unexpected keys: {len(unexpected)} (should be 0)")
        
        self.model = self.model.eval().cuda().half()

        # Transform: BICUBIC resize to 448, CenterCrop, ImageNet normalize
        # Must use BICUBIC — this matches CONCH v1.5 training pipeline
        self.transform = transforms.Compose([
            transforms.Resize(
                448,
                interpolation=transforms.InterpolationMode.BICUBIC
            ),
            transforms.CenterCrop(448),
            transforms.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.485, 0.456, 0.406),
                std =(0.229, 0.224, 0.225)
            )
        ])

        used = torch.cuda.memory_allocated() / 1e9
        print(f"CONCH v1.5 loaded. VRAM used: {used:.2f} GB")
        print(f"Output dim: 768")

    def extract_batch(self, pil_images: list) -> torch.Tensor:
        batch = torch.stack([self.transform(img) for img in pil_images])
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            with torch.inference_mode():
                # forward_features returns (B, N_tokens, 768)
                features = self.model.forward_features(batch.cuda())
                # Mean pool all patch tokens
                # ViT-L with img=448, patch=16 → (448/16)² = 784 tokens
                # Token 0 is CLS — skip it, mean pool tokens 1:
                embeddings = features[:, 1:, :].mean(dim=1)  # (B, 768)
        return embeddings.cpu().float()

In [11]:
# Extraction function 
def process_sample(sample_name, extractor, config):
    sample_dir = os.path.join(config["input_png_root"], sample_name)
    out_path   = os.path.join(config["output_root"], f"{sample_name}_conchv15.pt")

    if not os.path.exists(sample_dir):
        print(f"[SKIP] Directory not found: {sample_dir}")
        return

    if os.path.exists(out_path):
        saved = torch.load(out_path, map_location="cpu", weights_only=False)
        print(f"[SKIP] {sample_name} already done ({len(saved['patch_names'])} patches)")
        return

    patches = sorted(glob.glob(os.path.join(sample_dir, "*.png")))
    if not patches:
        print(f"[SKIP] No PNGs found in {sample_dir}")
        return

    print(f"\nProcessing {sample_name}: {len(patches)} patches")

    all_embs, all_names = [], []
    batch_imgs, batch_names = [], []

    for patch_path in tqdm(patches, desc=sample_name):
        img = Image.open(patch_path).convert("RGB")
        batch_imgs.append(img)
        batch_names.append(os.path.splitext(os.path.basename(patch_path))[0])

        if len(batch_imgs) == config["batch_size"]:
            all_embs.append(extractor.extract_batch(batch_imgs))
            all_names.extend(batch_names)
            batch_imgs, batch_names = [], []

    if batch_imgs:  # leftover partial batch
        all_embs.append(extractor.extract_batch(batch_imgs))
        all_names.extend(batch_names)

    torch.save({
        "embeddings":  torch.cat(all_embs, dim=0),  # (N, 768)
        "patch_names": all_names,
        "model":       "CONCH-v1.5",
        "patient":     sample_name
    }, out_path)
    print(f"Saved {len(all_names)} embeddings -> {out_path}")

In [12]:
# Run 
extractor = CONCHv15Extractor()

for sample_name in CONFIG["samples"]:
    torch.cuda.empty_cache()
    try:
        process_sample(sample_name, extractor, CONFIG)
    except Exception as e:
        print(f"[ERROR] {sample_name}: {e}")

Weights downloaded to: /root/.cache/huggingface/hub/models--MahmoodLab--conchv1_5/snapshots/3e5766a5d1500d53c73c03005e24c30c1f27be13/pytorch_model_vision.bin
Total keys in file : 352
Sample keys        : ['trunk.cls_token', 'trunk.pos_embed', 'trunk.patch_embed.proj.weight', 'trunk.patch_embed.proj.bias', 'trunk.blocks.0.norm1.weight']
Trunk keys extracted: 342
Missing keys   : 0    (should be 0)
Unexpected keys: 0 (should be 0)
CONCH v1.5 loaded. VRAM used: 0.61 GB
Output dim: 768

Processing IU_PDA_HM11: 3931 patches


IU_PDA_HM11: 100%|██████████| 3931/3931 [02:55<00:00, 22.36it/s]


Saved 3931 embeddings -> /kaggle/working/Features/CONCHv1_5/IU_PDA_HM11_conchv15.pt

Processing IU_PDA_HM13: 2182 patches


IU_PDA_HM13: 100%|██████████| 2182/2182 [01:42<00:00, 21.19it/s]


Saved 2182 embeddings -> /kaggle/working/Features/CONCHv1_5/IU_PDA_HM13_conchv15.pt

Processing IU_PDA_T1: 3530 patches


...

Saved 2777 embeddings -> /kaggle/working/Features/CONCHv1_5/IU_PDA_T11_conchv15.pt

Processing IU_PDA_T3: 4354 patches


IU_PDA_T3: 100%|██████████| 4354/4354 [03:22<00:00, 21.49it/s]


Saved 4354 embeddings -> /kaggle/working/Features/CONCHv1_5/IU_PDA_T3_conchv15.pt

Processing IU_PDA_T4: 3621 patches


IU_PDA_T4: 100%|██████████| 3621/3621 [02:48<00:00, 21.47it/s]


Saved 3621 embeddings -> /kaggle/working/Features/CONCHv1_5/IU_PDA_T4_conchv15.pt


In [13]:
# Verification 
print("=" * 60)
print("SUMMARY")
print("=" * 60)

for sample_name in CONFIG["samples"]:
    out_path = os.path.join(CONFIG["output_root"], f"{sample_name}_conchv15.pt")
    if os.path.exists(out_path):
        data = torch.load(out_path, map_location="cpu", weights_only=False)
        emb  = data["embeddings"]
        print(f"{sample_name}: {emb.shape} | mean={emb.mean():.4f} | std={emb.std():.4f}")
    else:
        print(f"{sample_name}: NOT FOUND")

SUMMARY
IU_PDA_HM11: torch.Size([3931, 1024]) | mean=-0.0023 | std=0.9797
IU_PDA_HM13: torch.Size([2182, 1024]) | mean=-0.0022 | std=0.9677
IU_PDA_T1: torch.Size([3530, 1024]) | mean=-0.0014 | std=0.9896
IU_PDA_T11: torch.Size([2777, 1024]) | mean=-0.0001 | std=0.9784
IU_PDA_T3: torch.Size([4354, 1024]) | mean=-0.0022 | std=0.9744
IU_PDA_T4: torch.Size([3621, 1024]) | mean=-0.0031 | std=0.9838


In [14]:
# Load & Inspect
import torch.nn.functional as F

def inspect_embedding(sample_name: str, patch_index: int = 0):
    out_path = os.path.join(CONFIG["output_root"], f"{sample_name}_conchv15.pt")

    if not os.path.exists(out_path):
        print(f"File not found: {out_path}")
        return

    data        = torch.load(out_path, map_location="cpu", weights_only=False)
    embeddings  = data["embeddings"]
    patch_names = data["patch_names"]

    print("=" * 60)
    print(f"Patient : {data['patient']}")
    print(f"Model   : {data['model']}")
    print("=" * 60)

    print(f"\n[FILE]")
    print(f"  Total patches    : {embeddings.shape[0]}")
    print(f"  Embedding dim    : {embeddings.shape[1]}")   # should be 768
    print(f"  Dtype            : {embeddings.dtype}")
    print(f"  File size on disk: {os.path.getsize(out_path)/1e6:.2f} MB")

    print(f"\n[MATRIX STATS]")
    print(f"  Global mean : {embeddings.mean():.6f}")
    print(f"  Global std  : {embeddings.std():.6f}")
    print(f"  Global min  : {embeddings.min():.6f}")
    print(f"  Global max  : {embeddings.max():.6f}")

    idx = min(patch_index, len(patch_names) - 1)
    emb = embeddings[idx]
    print(f"\n[PATCH #{idx}]")
    print(f"  Name          : {patch_names[idx]}")
    print(f"  Shape         : {emb.shape}")
    print(f"  Norm          : {emb.norm():.6f}")   # ~27 for 768-dim (√768 ≈ 27.7)
    print(f"  First 8 values: {emb[:8].tolist()}")

    print(f"\n[SANITY CHECKS]")
    print(f"  Dim is 768             : {'PASS' if embeddings.shape[1] == 768 else 'FAIL'}")
    print(f"  No NaNs                : {'PASS' if not torch.isnan(embeddings).any() else 'FAIL'}")
    zero_rows = (embeddings.abs().sum(dim=1) == 0).sum().item()
    print(f"  Zero embeddings        : {zero_rows} {'(PASS)' if zero_rows == 0 else '(WARN)'}")
    print(f"  Names/embeddings match : {'PASS' if len(patch_names) == embeddings.shape[0] else 'FAIL'}")
    print(f"  Dtype is float32       : {'PASS' if embeddings.dtype == torch.float32 else 'FAIL'}")

    # Norms should be ~27.7 (NOT ~1.0 — no normalize=True here unlike CONCH v1)
    norms = embeddings.norm(dim=1)
    print(f"  Mean norm              : {norms.mean():.4f}  (expected ~27)")

    if embeddings.shape[0] >= 2:
        e1 = F.normalize(embeddings[0].unsqueeze(0), dim=1)
        e2 = F.normalize(embeddings[1].unsqueeze(0), dim=1)
        sim = (e1 * e2).sum().item()
        print(f"\n[SIMILARITY]")
        print(f"  Cosine sim (patch 0 vs 1): {sim:.4f}")
        print(f"  (healthy range: 0.5–0.95 for same-tissue patches)")

inspect_embedding(CONFIG["samples"][0], patch_index=0)

Patient : IU_PDA_HM11
Model   : CONCH-v1.5

[FILE]
  Total patches    : 3931
  Embedding dim    : 1024
  Dtype            : torch.float32
  File size on disk: 16.26 MB

[MATRIX STATS]
  Global mean : -0.002265
  Global std  : 0.979735
  Global min  : -8.800528
  Global max  : 6.527224

[PATCH #0]
  Name          : IU_PDA_HM11_patch-000001_50_102
  Shape         : torch.Size([1024])
  Norm          : 30.672699
  First 8 values: [-0.9142009019851685, 0.28106310963630676, -0.538958728313446, -1.0959478616714478, 0.9650006294250488, 0.41539210081100464, -1.4013811349868774, 0.54474276304245]

[SANITY CHECKS]
  Dim is 768             : FAIL
  No NaNs                : PASS
  Zero embeddings        : 0 (PASS)
  Names/embeddings match : PASS
  Dtype is float32       : PASS
  Mean norm              : 31.3481  (expected ~27)

[SIMILARITY]
  Cosine sim (patch 0 vs 1): 0.7449
  (healthy range: 0.5–0.95 for same-tissue patches)
